In [28]:
import os
import pandas as pd
import re
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pickle # lưu model sau khi training

# Import tf-idf và cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Import success!")

Import success!


In [29]:
def load_data():
    """Kết nối DB"""
    load_dotenv()
    
    try:
        connection_string = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_DATABASE')}"
        engine = create_engine(connection_string)
        print("Kết nối CSDL PostgreSQL thành công!")
    except Exception as e:
        print(f"LỖI: Không thể kết nối CSDL. Lỗi: {e}")
        return None

    # Các câu lệnh SQL
    sql_movies = "SELECT id, title, summary as description FROM Movies"
    sql_genres = "SELECT movie_id, STRING_AGG(G.name, ', ') AS genres FROM Movie_Genres MG JOIN Genres G ON MG.genre_id = G.id GROUP BY MG.movie_id"
    sql_actors = "SELECT movie_id, STRING_AGG(A.name, ', ') AS actors FROM Movie_Actors MA JOIN Actors A ON MA.actor_id = A.id GROUP BY MA.movie_id"
    sql_directors = "SELECT movie_id, STRING_AGG(D.name, ', ') AS directors FROM Movie_Directors MD JOIN Directors D ON MD.director_id = D.id GROUP BY MD.movie_id"

    try:
        print("Đang tải dữ liệu (movies, genres, actors, directors)...")
        df_movies = pd.read_sql(sql_movies, engine)
        df_genres = pd.read_sql(sql_genres, engine)
        df_actors = pd.read_sql(sql_actors, engine)
        df_directors = pd.read_sql(sql_directors, engine)

        # Gộp (Merge) tất cả
        df_full = pd.merge(df_movies, df_genres, left_on='id', right_on='movie_id', how='left')
        df_full = pd.merge(df_full, df_actors, left_on='id', right_on='movie_id', how='left')
        df_full = pd.merge(df_full, df_directors, left_on='id', right_on='movie_id', how='left')
        
        df_full = df_full.drop(columns=['movie_id_x', 'movie_id_y', 'movie_id'])
        print("Tải và gộp dữ liệu thành công!")
        return df_full

    except Exception as e:
        print(f"LỖI: Không thể tải dữ liệu. Lỗi: {e}")
        return None

In [30]:
def clean_text(text):
    """Hàm dọn dẹp (xóa dấu cách, viết thường)"""
    if text is None:
        return ""
    names = [re.sub(r'\s+', '', name).lower() for name in text.split(', ')]
    return " ".join(names)

In [31]:
def create_soup(data_row):
    """Hàm trộn các nguyên liệu (genres, actors, directors)"""
    genres = clean_text(data_row['genres'])
    actors = clean_text(data_row['actors'])
    directors = clean_text(data_row['directors'])
    description = clean_text(data_row['description'])
    return f"{genres} {actors} {directors} {description}"

In [32]:
# Load data from database
data = load_data()

data.head()

Kết nối CSDL PostgreSQL thành công!
Đang tải dữ liệu (movies, genres, actors, directors)...
Tải và gộp dữ liệu thành công!


,id,title,description,genres,actors,directors
0,278,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"Drama, Crime","Clancy Brown, Tim Robbins, Morgan Freeman, Bob...",Frank Darabont
1,1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"Fantasy, Animation, Action, Thriller","Natsuki Hanae, Hiro Shimono, Yoshitsugu Matsuo...",Haruo Sotozaki
2,1084242,Zootopia 2,After cracking the biggest case in Zootopia's ...,"Adventure, Animation, Comedy, Mystery, Family","Andy Samberg, Ginnifer Goodwin, Jason Bateman,...",Jared Bush
3,507244,Afterburn,Set against the backdrop of a postapocalyptic ...,"Action, Comedy, Science Fiction","Dave Bautista, Samuel L. Jackson, Daniel Bernh...",J.J. Perry
4,299536,Avengers: Infinity War,As the Avengers and their allies have continue...,"Adventure, Action, Science Fiction","Josh Brolin, Chris Evans, Robert Downey Jr., C...",Joe Russo


In [33]:
# Tiền xử lý
data['genres'] = data['genres'].fillna('')
data['actors'] = data['actors'].fillna('')
data['directors'] = data['directors'].fillna('')
data['description'] = data['description'].fillna('')

# Tạo cột soup
data['soup'] = data.apply(create_soup, axis=1)

print('success!')

data[['id', 'title', 'soup']].head()

success!


,id,title,soup
0,278,The Shawshank Redemption,drama crime clancybrown timrobbins morganfreem...
1,1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,fantasy animation action thriller natsukihanae...
2,1084242,Zootopia 2,adventure animation comedy mystery family andy...
3,507244,Afterburn,action comedy sciencefiction davebautista samu...
4,299536,Avengers: Infinity War,adventure action sciencefiction joshbrolin chr...


In [34]:
# Tính tf-idf matrix
vectorize = TfidfVectorizer(stop_words='english')

tfidf_matrix = vectorize.fit_transform(data['soup'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (1554, 16936)


In [35]:
# import SVD để giảm chiều dữ liệu
from sklearn.decomposition import TruncatedSVD

In [36]:
svd = TruncatedSVD(n_components=100, random_state=42)

latent_matrix = svd.fit_transform(tfidf_matrix)
print("Latent matrix shape:", latent_matrix.shape)

Latent matrix shape: (1554, 100)


In [37]:
# Tinh cosine similarity
cosine_sim_svd = cosine_similarity(latent_matrix, latent_matrix)

In [38]:
print('Shape:', cosine_sim_svd.shape)

print(cosine_sim_svd[0:5, 0:5])

Shape: (1554, 1554)
[[ 1.         -0.07905525 -0.05683015  0.05326059 -0.01811159]
 [-0.07905525  1.          0.00891697  0.07936852  0.05577434]
 [-0.05683015  0.00891697  1.         -0.00636636  0.01418449]
 [ 0.05326059  0.07936852 -0.00636636  1.          0.03603018]
 [-0.01811159  0.05577434  0.01418449  0.03603018  1.        ]]


In [39]:
# Lưu bảng tra cứu id -> index
indices = pd.Series(data.index, index=data['id']).drop_duplicates()
indices.head()

id
278        0
1311031    1
1084242    2
507244     3
299536     4
dtype: int64

In [ ]:
# Hàm gợi ý
def get_recommendations(movie_id, cosine_sim_matrix = cosine_sim_svd, data=data):
    try:
        idx = indices[movie_id]
        sim_scores = list(enumerate(cosine_sim_matrix[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

        sim_scores = sim_scores[1:20]

        movie_indices = [i[0] for i in sim_scores]

        return data['title'].iloc[movie_indices]
    except KeyError:
        return f"Lỗi: Không tìm thấy ID phim {movie_id} trong dữ liệu."
    except Exception as e:
        return f"Lỗi: {e}"

print("Already trained model")

Already trained model, ready to recommend!


In [41]:
test_movie_id = 155

print(f"--- Gợi ý cho phim (ID: {test_movie_id}) ---")
recommendations = get_recommendations(test_movie_id)
print(recommendations)

--- Gợi ý cho phim (ID: 155) ---
1158                 Batman Begins
1278                  The Prestige
743          The Dark Knight Rises
255                   Interstellar
603           Billion Dollar Brain
1416               American Psycho
1012                  Curtain Call
274                      Bewitched
463                 Public Enemies
31                     Oppenheimer
964                      Inception
1535                       Memento
651                      Amsterdam
1042    10 Things I Hate About You
798                      Dillinger
600         Muzzle: City of Wolves
746                         Cars 2
221                 Inside Furioza
830                        Madness
Name: title, dtype: object


## Đánh giá hệ gợi ý (offline)

Vì đây là content-based recommender (không có nhãn train/test kiểu supervised), ta dùng đánh giá offline theo **genre overlap**:

- **Precision@K**: tỷ lệ phim trong top-K gợi ý có chung ít nhất 1 thể loại với phim truy vấn.
- **HitRate@K**: với mỗi phim truy vấn, top-K có ít nhất 1 phim liên quan hay không.

> Đây là đánh giá proxy nhanh. Nếu có dữ liệu tương tác người dùng (view/click/rating), nên đánh giá thêm bằng Recall@K, MAP@K, NDCG@K.

In [ ]:
import numpy as np


def _to_genre_set(x):
    if pd.isna(x) or not str(x).strip():
        return set()
    return {g.strip().lower() for g in str(x).split(',') if g.strip()}


def evaluate_content_recommender(data, cosine_sim_matrix, indices, k=10):
    """Đánh giá recommender bằng genre overlap.

    Trả về:
    - precision_at_k: trung bình tỷ lệ item liên quan trong top-K
    - hit_rate_at_k: tỷ lệ query có ít nhất 1 item liên quan trong top-K
    - evaluated_queries: số lượng phim được dùng để đánh giá
    """
    if 'genres' not in data.columns:
        raise ValueError("Data phải có cột 'genres' để đánh giá.")

    genre_sets = data['genres'].apply(_to_genre_set)
    total_precision = 0.0
    total_hits = 0
    total_queries = 0

    for movie_id in data['id']:
        try:
            idx = indices[movie_id]
        except KeyError:
            continue

        query_genres = genre_sets.iloc[idx]
        if not query_genres:
            # Bỏ qua phim không có genre
            continue

        sim_scores = list(enumerate(cosine_sim_matrix[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:k+1]  # bỏ chính nó

        if not sim_scores:
            continue

        rec_indices = [i for i, _ in sim_scores]

        relevant_count = 0
        for rec_idx in rec_indices:
            if query_genres & genre_sets.iloc[rec_idx]:
                relevant_count += 1

        precision_q = relevant_count / k
        hit_q = 1 if relevant_count > 0 else 0

        total_precision += precision_q
        total_hits += hit_q
        total_queries += 1

    if total_queries == 0:
        return {
            'precision_at_k': 0.0,
            'hit_rate_at_k': 0.0,
            'evaluated_queries': 0
        }

    return {
        'precision_at_k': total_precision / total_queries,
        'hit_rate_at_k': total_hits / total_queries,
        'evaluated_queries': total_queries
    }


K = 10
metrics = evaluate_content_recommender(data, cosine_sim_svd, indices, k=K)

print(f"Evaluated queries: {metrics['evaluated_queries']}")
print(f"Precision@{K}: {metrics['precision_at_k']:.4f}")
print(f"HitRate@{K}: {metrics['hit_rate_at_k']:.4f}")